# Advanced Narrative Surprise Analyzer

**Authors:** Devashish Juyal, Yuganshi Agrawal  
**University of Michigan**

This notebook implements an advanced methodology for quantifying narrative surprise in stories using:
- OpenAI's `text-embedding-3-large` or `text-embedding-3-small` for semantic embeddings
- Multiple divergence metrics: Cosine distance, Euclidean distance, KL divergence
- Comprehensive output storage including raw embeddings
- Advanced visualizations: line plots, heatmaps, histograms
- Multi-format export: CSV, JSONL, Parquet

## 1. Setup and Configuration

In [ ]:
# Core imports
import os
import json
import math
import time
import hashlib
from glob import glob
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Any
from dataclasses import dataclass, asdict

# Data manipulation
import numpy as np
import pandas as pd

# NLP
from nltk.tokenize import sent_tokenize
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# OpenAI
import openai
import backoff

# Scientific computing
from scipy.spatial.distance import cosine, euclidean
from scipy.special import rel_entr, softmax
from scipy.ndimage import gaussian_filter1d
from scipy.signal import find_peaks

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch

# Progress tracking
from tqdm.notebook import tqdm

# Environment
from dotenv import load_dotenv

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print("✅ All imports successful")

In [ ]:
# Load environment variables
load_dotenv()

# Initialize OpenAI client
api_key = os.getenv('OPENAI_API_KEY')
if not api_key:
    raise ValueError("OPENAI_API_KEY not found in environment variables")

client = openai.OpenAI(api_key=api_key)

# Configuration
@dataclass
class Config:
    """Configuration for the surprise analyzer."""
    # Paths
    stories_dir: str = "./StoryScraper/stories"
    output_dir: str = "./outputs/advanced"
    
    # Model settings
    chat_model: str = "gpt-4o-mini"  # For predictions/summaries
    embedding_model: str = "text-embedding-3-large"  # or "text-embedding-3-small"
    embedding_dimensions: int = 3072  # 3072 for large, 1536 for small
    
    # Chunking parameters
    chunk_size: int = 300  # Target words per chunk
    tolerance: int = 30    # Word tolerance for sentence boundary alignment
    min_sentence_words: int = 5  # Minimum words to keep a sentence separate
    max_sentence_words: int = 200  # Maximum words before splitting
    
    # Processing
    max_completion_tokens: int = 400
    save_embeddings: bool = True  # Whether to save raw embeddings
    
config = Config()

# Create output directory
os.makedirs(config.output_dir, exist_ok=True)

print(f"📁 Stories directory: {config.stories_dir}")
print(f"📁 Output directory: {config.output_dir}")
print(f"🤖 Chat model: {config.chat_model}")
print(f"🔢 Embedding model: {config.embedding_model}")

## 2. Data Structures

In [ ]:
@dataclass
class ChunkResult:
    """Stores the analysis result for a single story chunk."""
    story_id: str
    story_title: str
    chunk_index: int
    position: float  # 0-1 normalized position in story
    
    # Text content
    passage: str
    bau_prediction: str  # Business-as-usual prediction
    actual_summary: str  # Summary of actual next event
    
    # Embeddings (as lists for JSON serialization)
    prediction_embedding: List[float]
    actual_embedding: List[float]
    
    # Surprise metrics
    cosine_distance: float
    euclidean_distance: float
    kl_divergence: float
    
    # Metadata
    passage_word_count: int
    chunk_word_start: int
    chunk_word_end: int
    
    def to_dict(self, include_embeddings: bool = True) -> dict:
        """Convert to dictionary for export."""
        d = asdict(self)
        if not include_embeddings:
            d.pop('prediction_embedding', None)
            d.pop('actual_embedding', None)
        return d


@dataclass  
class StoryAnalysis:
    """Aggregated analysis for an entire story."""
    story_id: str
    story_title: str
    total_words: int
    total_chunks: int
    
    # Aggregate metrics
    mean_cosine_distance: float
    std_cosine_distance: float
    mean_euclidean_distance: float
    std_euclidean_distance: float
    mean_kl_divergence: float
    std_kl_divergence: float
    
    # Peak detection
    num_detected_twists: int
    twist_positions: List[float]
    
    # Genre (if available)
    genre: str = "thriller"

print("✅ Data structures defined")

## 3. Text Processing Functions

In [ ]:
def generate_story_id(filepath: str) -> str:
    """Generate a unique story ID from filepath."""
    return hashlib.md5(filepath.encode()).hexdigest()[:12]


def fuse_short_sentences(sentences: List[str], 
                         word_limit: int = 5, 
                         max_len: int = 200, 
                         split_len: int = 160) -> List[str]:
    """
    Merge tiny sentences into previous ones and split ultra-long sentences.
    Keeps sentence boundaries reasonable for chunking.
    """
    fused = []

    for s in sentences:
        words = s.split()

        # Too short -> merge into previous
        if len(words) < word_limit and fused:
            fused[-1] = fused[-1] + " " + s
            continue

        # Too long -> split into ~split_len word pieces
        if len(words) > max_len:
            n_parts = (len(words) // split_len) + 1
            step = len(words) // n_parts
            for i in range(n_parts):
                part = " ".join(words[i*step:(i+1)*step])
                if part.strip():
                    fused.append(part)
            # Tail
            tail = " ".join(words[n_parts*step:])
            if tail.strip():
                fused.append(tail)
            continue

        # Normal sentence
        fused.append(s)

    return fused


def closest_sentence_index(sentence_indexes: List[int], target: int) -> Optional[int]:
    """Return the sentence-end word index closest to 'target'."""
    if not sentence_indexes:
        return None
    return min(sentence_indexes, key=lambda x: abs(target - x))


def sentence_indexes_within_range(sentence_indexes: List[int], 
                                   target: int, 
                                   range_val: int = 30) -> List[int]:
    """Return all sentence-end word indices within ±range_val of 'target'."""
    return [idx for idx in sentence_indexes if abs(target - idx) <= range_val]


def tokenize_story(filepath: str) -> Tuple[List[str], List[int], int]:
    """
    Read a story, tokenize to sentences, and compute cumulative word indices.
    
    Returns:
        sentences: List of sentences (with empty string at index 0)
        word_divides: Cumulative word count at end of each sentence
        total_words: Total word count
    """
    with open(filepath, mode='r', encoding='utf-8') as f:
        text = f.read()

    raw_sentences = sent_tokenize(text)
    sentences = fuse_short_sentences(
        raw_sentences, 
        word_limit=config.min_sentence_words,
        max_len=config.max_sentence_words
    )

    # Build cumulative word indices
    all_sentences = ['']  # Dummy at index 0
    word_divides = [0]
    
    cumulative = 0
    for s in sentences:
        word_count = len(s.split())
        cumulative += word_count
        all_sentences.append(s)
        word_divides.append(cumulative)

    return all_sentences, word_divides, cumulative


def create_chunks(sentences: List[str], 
                  word_divides: List[int], 
                  total_words: int,
                  chunk_size: int = 300,
                  tolerance: int = 30) -> List[Tuple[int, int, int, int]]:
    """
    Create chunks of ~chunk_size words, aligned to sentence boundaries.
    
    Returns:
        List of (start_sentence_idx, end_sentence_idx, word_start, word_end)
    """
    sentence_ends = word_divides[1:]  # Skip the leading 0
    
    # Generate ideal boundary points
    ideals = list(range(chunk_size, total_words, chunk_size))
    ideals.append(total_words)
    
    # Snap to actual sentence ends
    chosen_points = []
    for target in ideals:
        if target >= total_words:
            chosen_points.append(total_words)
            continue
            
        candidates = sentence_indexes_within_range(sentence_ends, target, tolerance)
        if candidates:
            point = min(candidates, key=lambda x: abs(x - target))
        else:
            point = closest_sentence_index(sentence_ends, target)
        chosen_points.append(point)
    
    # Remove duplicates and sort
    chosen_points = sorted(set(chosen_points))
    
    # Convert to chunk tuples
    chunks = []
    last_point = 0
    for point in chosen_points:
        start_sentence = word_divides.index(last_point) + 1
        end_sentence = word_divides.index(point) if point in word_divides else len(word_divides) - 1
        chunks.append((start_sentence, end_sentence, last_point, point))
        last_point = point
    
    return chunks


def get_chunk_text(sentences: List[str], 
                   start_idx: int, 
                   end_idx: int) -> str:
    """Extract text for a chunk given sentence indices."""
    return ' '.join(sentences[start_idx:end_idx+1]).strip()

print("✅ Text processing functions defined")

## 4. OpenAI API Functions

In [ ]:
# System prompts
SUMMARY_SYSTEM_PROMPT = """
You are summarizing short stories or story excerpts.
Each summary must be a single, self-contained present-tense sentence that describes what is happening in the scene.
Do not use phrases like 'in this story' or 'the passage describes'—write as though you are inside the narrative.
Focus on the key action or emotion, and omit background detail.
Enclose the summary in <summary> and </summary> tags.
""".strip()

PREDICTION_SYSTEM_PROMPT = """
You are predicting the next event in a short story based on what has already happened.
Write one confident, present-tense sentence about what occurs next in the narrative, without hedging or speculation.
Do not refer to yourself or the text; stay within the story world.
This is a "business-as-usual" prediction: predict what would typically happen next based on story conventions.
Enclose the prediction in <prediction> and </prediction> tags.
""".strip()

# Few-shot examples
SUMMARY_FEWSHOTS = [
    {
        "role": "user",
        "content": """Read the short story passage below and summarize it in one present-tense sentence.

<passage>
The night was silent except for the drip of rain from the gutter. Mara pressed herself against the door, listening to the footsteps outside. Someone—or something—was circling the house.
</passage>"""
    },
    {
        "role": "assistant",
        "content": "<summary>Mara hides in her home as an unseen figure stalks outside in the rain.</summary>"
    },
    {
        "role": "user",
        "content": """Read the short story passage below and summarize it in one present-tense sentence.

<passage>
David clutched the old photograph, its corners frayed and stained with blood. He couldn't remember taking it—but the smiling face beside him was his own.
</passage>"""
    },
    {
        "role": "assistant",
        "content": "<summary>David realizes the photograph shows an alternate version of himself he cannot recall.</summary>"
    }
]

PREDICTION_FEWSHOTS = [
    {
        "role": "user",
        "content": """Based on the short story passage below, predict what happens next in one confident, present-tense sentence.

<passage>
Mara hides in the darkness, clutching the kitchen knife. The footsteps stop just outside the window.
</passage>"""
    },
    {
        "role": "assistant",
        "content": "<prediction>The intruder shatters the window, forcing Mara to fight for her life.</prediction>"
    },
    {
        "role": "user",
        "content": """Based on the short story passage below, predict what happens next in one confident, present-tense sentence.

<passage>
The lights flickered, and the photograph in David's hand began to fade, the other man's face slowly vanishing.
</passage>"""
    },
    {
        "role": "assistant",
        "content": "<prediction>David feels his own body begin to dissolve as the photo disappears.</prediction>"
    }
]


@backoff.on_exception(
    backoff.expo,
    (openai.RateLimitError, openai.APIError),
    max_time=120,
    max_tries=5
)
def call_chat_api(messages: List[dict], 
                  model: str = None, 
                  max_tokens: int = None) -> str:
    """Call OpenAI chat API with exponential backoff."""
    model = model or config.chat_model
    max_tokens = max_tokens or config.max_completion_tokens
    
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        max_completion_tokens=max_tokens
    )
    return response.choices[0].message.content


@backoff.on_exception(
    backoff.expo,
    (openai.RateLimitError, openai.APIError),
    max_time=120,
    max_tries=5
)
def get_embedding(text: str, model: str = None) -> List[float]:
    """Get embedding for text using OpenAI's embedding API."""
    model = model or config.embedding_model
    
    response = client.embeddings.create(
        model=model,
        input=text,
        encoding_format="float"
    )
    return response.data[0].embedding


@backoff.on_exception(
    backoff.expo,
    (openai.RateLimitError, openai.APIError),
    max_time=120,
    max_tries=5
)
def get_embeddings_batch(texts: List[str], model: str = None) -> List[List[float]]:
    """Get embeddings for multiple texts in a single API call."""
    model = model or config.embedding_model
    
    response = client.embeddings.create(
        model=model,
        input=texts,
        encoding_format="float"
    )
    return [item.embedding for item in response.data]

print("✅ API functions defined")

In [ ]:
import re

def parse_summary(text: str) -> str:
    """Extract summary from model response."""
    if not text:
        return ""
    
    match = re.search(r"<\s*summary\s*>(.*?)<\s*/\s*summary\s*>", text, re.IGNORECASE | re.DOTALL)
    if match:
        return match.group(1).strip()
    return re.sub(r"<.*?>", "", text).strip()


def parse_prediction(text: str) -> str:
    """Extract prediction from model response."""
    if not text:
        return ""
    
    match = re.search(r"<\s*prediction\s*>(.*?)<\s*/\s*prediction\s*>", text, re.IGNORECASE | re.DOTALL)
    if match:
        return match.group(1).strip()
    return re.sub(r"<.*?>", "", text).strip()


def generate_prediction(passage: str, recap: str = "") -> str:
    """Generate BAU prediction for what happens next."""
    context = f"\n\nStory so far: {recap}" if recap else ""
    
    messages = [
        {"role": "system", "content": PREDICTION_SYSTEM_PROMPT},
        *PREDICTION_FEWSHOTS,
        {
            "role": "user",
            "content": f"""Based on the short story passage below, predict what happens next in one confident, present-tense sentence.{context}

<passage>
{passage}
</passage>"""
        }
    ]
    
    response = call_chat_api(messages)
    return parse_prediction(response)


def generate_summary(passage: str) -> str:
    """Generate present-tense summary of the passage."""
    messages = [
        {"role": "system", "content": SUMMARY_SYSTEM_PROMPT},
        *SUMMARY_FEWSHOTS,
        {
            "role": "user",
            "content": f"""Read the short story passage below and summarize it in one present-tense sentence.

<passage>
{passage}
</passage>"""
        }
    ]
    
    response = call_chat_api(messages)
    return parse_summary(response)

print("✅ Generation and parsing functions defined")

## 5. Divergence Metrics

In [ ]:
def compute_cosine_distance(emb1: List[float], emb2: List[float]) -> float:
    """
    Compute cosine distance between two embeddings.
    Cosine distance = 1 - cosine_similarity
    Range: [0, 2] where 0 = identical, 2 = opposite
    """
    return cosine(emb1, emb2)


def compute_euclidean_distance(emb1: List[float], emb2: List[float]) -> float:
    """
    Compute Euclidean distance between two embeddings.
    Range: [0, ∞) where 0 = identical
    """
    return euclidean(emb1, emb2)


def compute_kl_divergence(emb1: List[float], emb2: List[float]) -> float:
    """
    Compute KL divergence between two embeddings after normalizing to probability distributions.
    Uses softmax normalization to convert embeddings to valid probability distributions.
    
    Returns: KL(P || Q) where P = softmax(emb1), Q = softmax(emb2)
    Range: [0, ∞) where 0 = identical distributions
    """
    # Convert to numpy arrays
    arr1 = np.array(emb1)
    arr2 = np.array(emb2)
    
    # Apply softmax to get probability distributions
    p = softmax(arr1)
    q = softmax(arr2)
    
    # Add small epsilon to avoid log(0)
    eps = 1e-10
    q = np.clip(q, eps, 1.0)
    
    # Compute KL divergence: sum(P * log(P/Q))
    kl = np.sum(rel_entr(p, q))
    
    return float(kl)


def compute_all_metrics(emb_prediction: List[float], 
                        emb_actual: List[float]) -> Dict[str, float]:
    """
    Compute all divergence metrics between prediction and actual embeddings.
    """
    return {
        'cosine_distance': compute_cosine_distance(emb_prediction, emb_actual),
        'euclidean_distance': compute_euclidean_distance(emb_prediction, emb_actual),
        'kl_divergence': compute_kl_divergence(emb_prediction, emb_actual)
    }


# Test the metrics
print("Testing divergence metrics...")
test_emb1 = [0.1, 0.2, 0.3, 0.4]
test_emb2 = [0.1, 0.2, 0.3, 0.5]
test_emb3 = [-0.1, -0.2, -0.3, -0.4]

print(f"Similar vectors: {compute_all_metrics(test_emb1, test_emb2)}")
print(f"Opposite vectors: {compute_all_metrics(test_emb1, test_emb3)}")
print("✅ Divergence metrics defined and tested")

## 6. Main Processing Pipeline

In [ ]:
def process_story(filepath: str, 
                  verbose: bool = True) -> Tuple[List[ChunkResult], StoryAnalysis]:
    """
    Process a single story through the complete surprise analysis pipeline.
    
    For each chunk:
    1. Generate BAU prediction for what happens next
    2. Generate summary of the actual next chunk
    3. Embed both using text-embedding-3
    4. Compute all divergence metrics
    
    Returns:
        chunk_results: List of ChunkResult objects
        story_analysis: Aggregated StoryAnalysis object
    """
    # Setup
    story_title = Path(filepath).stem
    story_id = generate_story_id(filepath)
    
    if verbose:
        print(f"\n{'='*60}")
        print(f"Processing: {story_title}")
        print(f"Story ID: {story_id}")
        print(f"{'='*60}")
    
    # Tokenize and chunk
    sentences, word_divides, total_words = tokenize_story(filepath)
    chunks = create_chunks(
        sentences, word_divides, total_words,
        chunk_size=config.chunk_size,
        tolerance=config.tolerance
    )
    
    if verbose:
        print(f"Total words: {total_words}")
        print(f"Total sentences: {len(sentences) - 1}")
        print(f"Total chunks: {len(chunks)}")
    
    chunk_results = []
    recap_summaries = []  # Running recap for context
    
    # Process each chunk (except the last one, since we need "next" chunk)
    iterator = tqdm(range(len(chunks) - 1), desc="Processing chunks") if verbose else range(len(chunks) - 1)
    
    for i in iterator:
        start_sent, end_sent, word_start, word_end = chunks[i]
        next_start, next_end, next_word_start, next_word_end = chunks[i + 1]
        
        # Get current chunk text
        current_passage = get_chunk_text(sentences, start_sent, end_sent)
        
        # Get next chunk text (the "actual" next event)
        next_passage = get_chunk_text(sentences, next_start, next_end)
        
        # Build recap from previous summaries (limit to ~500 words)
        recap = ' '.join(recap_summaries[-5:]) if recap_summaries else ""
        if len(recap.split()) > 500:
            recap = ' '.join(recap.split()[-500:])
        
        # Generate BAU prediction
        bau_prediction = generate_prediction(current_passage, recap)
        
        # Generate summary of actual next event
        actual_summary = generate_summary(next_passage)
        
        # Add to recap
        current_summary = generate_summary(current_passage)
        recap_summaries.append(current_summary)
        
        # Get embeddings (batch for efficiency)
        embeddings = get_embeddings_batch([bau_prediction, actual_summary])
        emb_prediction = embeddings[0]
        emb_actual = embeddings[1]
        
        # Compute metrics
        metrics = compute_all_metrics(emb_prediction, emb_actual)
        
        # Calculate position (0-1)
        position = word_end / total_words
        
        # Create result object
        result = ChunkResult(
            story_id=story_id,
            story_title=story_title,
            chunk_index=i,
            position=round(position, 5),
            passage=current_passage,
            bau_prediction=bau_prediction,
            actual_summary=actual_summary,
            prediction_embedding=emb_prediction if config.save_embeddings else [],
            actual_embedding=emb_actual if config.save_embeddings else [],
            cosine_distance=round(metrics['cosine_distance'], 6),
            euclidean_distance=round(metrics['euclidean_distance'], 6),
            kl_divergence=round(metrics['kl_divergence'], 6),
            passage_word_count=len(current_passage.split()),
            chunk_word_start=word_start,
            chunk_word_end=word_end
        )
        
        chunk_results.append(result)
        
        if verbose and (i + 1) % 5 == 0:
            print(f"\n[Chunk {i}] Position: {position:.2%}")
            print(f"  Prediction: {bau_prediction[:80]}...")
            print(f"  Actual: {actual_summary[:80]}...")
            print(f"  Cosine dist: {metrics['cosine_distance']:.4f}")
    
    # Compute story-level aggregates
    cosine_dists = [r.cosine_distance for r in chunk_results]
    euclidean_dists = [r.euclidean_distance for r in chunk_results]
    kl_divs = [r.kl_divergence for r in chunk_results]
    
    # Detect peaks/twists
    if len(cosine_dists) > 3:
        smoothed = gaussian_filter1d(cosine_dists, sigma=1)
        peaks, _ = find_peaks(smoothed, prominence=0.05)
        twist_positions = [chunk_results[p].position for p in peaks]
    else:
        twist_positions = []
    
    story_analysis = StoryAnalysis(
        story_id=story_id,
        story_title=story_title,
        total_words=total_words,
        total_chunks=len(chunks),
        mean_cosine_distance=float(np.mean(cosine_dists)),
        std_cosine_distance=float(np.std(cosine_dists)),
        mean_euclidean_distance=float(np.mean(euclidean_dists)),
        std_euclidean_distance=float(np.std(euclidean_dists)),
        mean_kl_divergence=float(np.mean(kl_divs)),
        std_kl_divergence=float(np.std(kl_divs)),
        num_detected_twists=len(twist_positions),
        twist_positions=twist_positions
    )
    
    if verbose:
        print(f"\n{'='*60}")
        print(f"Story Analysis Complete: {story_title}")
        print(f"Mean Cosine Distance: {story_analysis.mean_cosine_distance:.4f}")
        print(f"Mean Euclidean Distance: {story_analysis.mean_euclidean_distance:.4f}")
        print(f"Mean KL Divergence: {story_analysis.mean_kl_divergence:.4f}")
        print(f"Detected Twists: {story_analysis.num_detected_twists}")
        print(f"{'='*60}")
    
    return chunk_results, story_analysis

print("✅ Processing pipeline defined")

## 7. Visualization Functions

In [ ]:
def plot_surprise_over_time(chunk_results: List[ChunkResult],
                            metric: str = 'cosine_distance',
                            show_smoothed: bool = True,
                            show_peaks: bool = True,
                            figsize: Tuple[int, int] = (12, 5)) -> plt.Figure:
    """
    Plot surprise metric over story progression.
    
    Args:
        chunk_results: List of ChunkResult objects
        metric: 'cosine_distance', 'euclidean_distance', or 'kl_divergence'
        show_smoothed: Whether to overlay smoothed curve
        show_peaks: Whether to highlight detected peaks
    """
    positions = [r.position for r in chunk_results]
    values = [getattr(r, metric) for r in chunk_results]
    title = chunk_results[0].story_title if chunk_results else "Story"
    
    fig, ax = plt.subplots(figsize=figsize)
    
    # Raw values
    ax.plot(positions, values, alpha=0.4, linewidth=1, label=f'Raw {metric}')
    
    # Smoothed curve
    if show_smoothed and len(values) > 3:
        smoothed = gaussian_filter1d(values, sigma=1.5)
        ax.plot(positions, smoothed, linewidth=2.5, color='darkred', label='Smoothed')
    
    # Peak detection
    if show_peaks and len(values) > 3:
        smoothed = gaussian_filter1d(values, sigma=1.5)
        peaks, _ = find_peaks(smoothed, prominence=0.05)
        if len(peaks) > 0:
            peak_positions = [positions[p] for p in peaks]
            peak_values = [smoothed[p] for p in peaks]
            ax.scatter(peak_positions, peak_values, 
                      color='gold', s=100, edgecolor='black', 
                      zorder=5, label='Detected Twists')
    
    ax.set_xlabel('Story Position (0 → 1)', fontsize=12)
    ax.set_ylabel(f'{metric.replace("_", " ").title()}', fontsize=12)
    ax.set_title(f'Narrative Surprise — {title}', fontsize=14)
    ax.legend(frameon=True)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, 1)
    
    plt.tight_layout()
    return fig


def plot_all_metrics(chunk_results: List[ChunkResult],
                     figsize: Tuple[int, int] = (14, 10)) -> plt.Figure:
    """
    Plot all three metrics in subplots for comparison.
    """
    positions = [r.position for r in chunk_results]
    title = chunk_results[0].story_title if chunk_results else "Story"
    
    metrics = [
        ('cosine_distance', 'Cosine Distance', 'steelblue'),
        ('euclidean_distance', 'Euclidean Distance', 'forestgreen'),
        ('kl_divergence', 'KL Divergence', 'darkorange')
    ]
    
    fig, axes = plt.subplots(3, 1, figsize=figsize, sharex=True)
    
    for ax, (metric, label, color) in zip(axes, metrics):
        values = [getattr(r, metric) for r in chunk_results]
        
        ax.plot(positions, values, alpha=0.4, linewidth=1, color=color)
        
        if len(values) > 3:
            smoothed = gaussian_filter1d(values, sigma=1.5)
            ax.plot(positions, smoothed, linewidth=2.5, color=color, label=label)
            
            # Peaks
            peaks, _ = find_peaks(smoothed, prominence=0.03)
            if len(peaks) > 0:
                ax.scatter([positions[p] for p in peaks],
                          [smoothed[p] for p in peaks],
                          color='red', s=60, edgecolor='black', zorder=5)
        
        ax.set_ylabel(label, fontsize=11)
        ax.grid(True, alpha=0.3)
        ax.legend(loc='upper right')
    
    axes[-1].set_xlabel('Story Position (0 → 1)', fontsize=12)
    axes[0].set_title(f'Multi-Metric Narrative Surprise — {title}', fontsize=14)
    
    plt.tight_layout()
    return fig


def plot_heatmap(story_analyses: List[StoryAnalysis],
                 metric: str = 'mean_cosine_distance',
                 figsize: Tuple[int, int] = (12, 8)) -> plt.Figure:
    """
    Plot heatmap of surprise metrics across multiple stories.
    """
    # Create DataFrame
    data = []
    for sa in story_analyses:
        data.append({
            'Story': sa.story_title[:25],  # Truncate long titles
            'Cosine': sa.mean_cosine_distance,
            'Euclidean': sa.mean_euclidean_distance,
            'KL': sa.mean_kl_divergence,
            'Twists': sa.num_detected_twists
        })
    
    df = pd.DataFrame(data).set_index('Story')
    
    # Normalize for heatmap
    df_norm = (df - df.min()) / (df.max() - df.min() + 1e-10)
    
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(df_norm, annot=df.round(3), fmt='', cmap='YlOrRd', 
                ax=ax, cbar_kws={'label': 'Normalized Value'})
    ax.set_title('Surprise Metrics Heatmap Across Stories', fontsize=14)
    
    plt.tight_layout()
    return fig


def plot_histogram_by_genre(story_analyses: List[StoryAnalysis],
                            metric: str = 'mean_cosine_distance',
                            figsize: Tuple[int, int] = (10, 6)) -> plt.Figure:
    """
    Plot histogram of surprise distribution, optionally grouped by genre.
    """
    values = [getattr(sa, metric) for sa in story_analyses]
    genres = [sa.genre for sa in story_analyses]
    
    df = pd.DataFrame({'value': values, 'genre': genres})
    
    fig, ax = plt.subplots(figsize=figsize)
    
    unique_genres = df['genre'].unique()
    if len(unique_genres) > 1:
        for genre in unique_genres:
            subset = df[df['genre'] == genre]['value']
            ax.hist(subset, bins=15, alpha=0.6, label=genre)
        ax.legend()
    else:
        ax.hist(values, bins=20, color='steelblue', edgecolor='black', alpha=0.7)
    
    ax.set_xlabel(metric.replace('_', ' ').title(), fontsize=12)
    ax.set_ylabel('Frequency', fontsize=12)
    ax.set_title(f'Distribution of {metric.replace("_", " ").title()} Across Stories', fontsize=14)
    ax.axvline(np.mean(values), color='red', linestyle='--', label=f'Mean: {np.mean(values):.3f}')
    ax.legend()
    
    plt.tight_layout()
    return fig


def plot_metric_correlation(chunk_results: List[ChunkResult],
                            figsize: Tuple[int, int] = (10, 8)) -> plt.Figure:
    """
    Plot correlation matrix between different surprise metrics.
    """
    df = pd.DataFrame([
        {
            'Cosine Distance': r.cosine_distance,
            'Euclidean Distance': r.euclidean_distance,
            'KL Divergence': r.kl_divergence,
            'Position': r.position
        }
        for r in chunk_results
    ])
    
    fig, ax = plt.subplots(figsize=figsize)
    corr = df.corr()
    sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, 
                ax=ax, fmt='.3f', square=True)
    ax.set_title('Correlation Between Surprise Metrics', fontsize=14)
    
    plt.tight_layout()
    return fig

print("✅ Visualization functions defined")

## 8. Export Functions

In [ ]:
def export_to_csv(chunk_results: List[ChunkResult],
                  filepath: str,
                  include_embeddings: bool = False) -> None:
    """
    Export chunk results to CSV.
    Note: Embeddings are excluded by default due to size.
    """
    data = [r.to_dict(include_embeddings=include_embeddings) for r in chunk_results]
    df = pd.DataFrame(data)
    df.to_csv(filepath, index=False)
    print(f"✅ Exported {len(chunk_results)} chunks to CSV: {filepath}")


def export_to_jsonl(chunk_results: List[ChunkResult],
                    filepath: str,
                    include_embeddings: bool = True) -> None:
    """
    Export chunk results to JSONL (one JSON object per line).
    Supports embeddings since JSONL handles large arrays well.
    """
    with open(filepath, 'w') as f:
        for r in chunk_results:
            json_str = json.dumps(r.to_dict(include_embeddings=include_embeddings))
            f.write(json_str + '\n')
    print(f"✅ Exported {len(chunk_results)} chunks to JSONL: {filepath}")


def export_to_parquet(chunk_results: List[ChunkResult],
                      filepath: str,
                      include_embeddings: bool = True) -> None:
    """
    Export chunk results to Parquet format.
    Efficient for large datasets with embeddings.
    """
    data = [r.to_dict(include_embeddings=include_embeddings) for r in chunk_results]
    df = pd.DataFrame(data)
    
    # Convert embedding lists to strings for Parquet compatibility
    if include_embeddings and 'prediction_embedding' in df.columns:
        df['prediction_embedding'] = df['prediction_embedding'].apply(json.dumps)
        df['actual_embedding'] = df['actual_embedding'].apply(json.dumps)
    
    df.to_parquet(filepath, index=False)
    print(f"✅ Exported {len(chunk_results)} chunks to Parquet: {filepath}")


def export_story_summaries(story_analyses: List[StoryAnalysis],
                           filepath: str) -> None:
    """
    Export story-level summary statistics to CSV.
    """
    data = [asdict(sa) for sa in story_analyses]
    df = pd.DataFrame(data)
    
    # Convert list columns to strings
    df['twist_positions'] = df['twist_positions'].apply(lambda x: json.dumps(x))
    
    df.to_csv(filepath, index=False)
    print(f"✅ Exported {len(story_analyses)} story summaries to CSV: {filepath}")


def export_all_formats(chunk_results: List[ChunkResult],
                       story_analysis: StoryAnalysis,
                       output_dir: str) -> Dict[str, str]:
    """
    Export results in all supported formats.
    
    Returns dict of format -> filepath mappings.
    """
    os.makedirs(output_dir, exist_ok=True)
    story_name = story_analysis.story_title
    
    paths = {}
    
    # CSV (without embeddings for readability)
    csv_path = os.path.join(output_dir, f"{story_name}_chunks.csv")
    export_to_csv(chunk_results, csv_path, include_embeddings=False)
    paths['csv'] = csv_path
    
    # JSONL (with embeddings)
    jsonl_path = os.path.join(output_dir, f"{story_name}_chunks.jsonl")
    export_to_jsonl(chunk_results, jsonl_path, include_embeddings=True)
    paths['jsonl'] = jsonl_path
    
    # Parquet (with embeddings)
    parquet_path = os.path.join(output_dir, f"{story_name}_chunks.parquet")
    export_to_parquet(chunk_results, parquet_path, include_embeddings=True)
    paths['parquet'] = parquet_path
    
    # Story summary
    summary_path = os.path.join(output_dir, f"{story_name}_summary.csv")
    export_story_summaries([story_analysis], summary_path)
    paths['summary'] = summary_path
    
    return paths

print("✅ Export functions defined")

## 9. Batch Processing

In [ ]:
def process_multiple_stories(story_files: List[str],
                             output_dir: str = None,
                             verbose: bool = True) -> Tuple[List[List[ChunkResult]], List[StoryAnalysis]]:
    """
    Process multiple stories and aggregate results.
    
    Returns:
        all_chunks: List of chunk results per story
        all_analyses: List of story analyses
    """
    output_dir = output_dir or config.output_dir
    os.makedirs(output_dir, exist_ok=True)
    
    all_chunks = []
    all_analyses = []
    
    for filepath in tqdm(story_files, desc="Processing stories"):
        try:
            chunks, analysis = process_story(filepath, verbose=verbose)
            all_chunks.append(chunks)
            all_analyses.append(analysis)
            
            # Export individual story results
            export_all_formats(chunks, analysis, output_dir)
            
        except Exception as e:
            print(f"❌ Error processing {filepath}: {e}")
            continue
    
    # Export aggregate summary
    if all_analyses:
        aggregate_path = os.path.join(output_dir, "all_stories_summary.csv")
        export_story_summaries(all_analyses, aggregate_path)
    
    return all_chunks, all_analyses


def generate_all_visualizations(chunk_results: List[ChunkResult],
                                 story_analysis: StoryAnalysis,
                                 output_dir: str = None,
                                 save_plots: bool = True) -> Dict[str, plt.Figure]:
    """
    Generate all visualizations for a story.
    """
    output_dir = output_dir or config.output_dir
    story_name = story_analysis.story_title
    
    figures = {}
    
    # 1. Surprise over time (main metric)
    fig = plot_surprise_over_time(chunk_results, metric='cosine_distance')
    figures['surprise_timeline'] = fig
    if save_plots:
        fig.savefig(os.path.join(output_dir, f"{story_name}_surprise_timeline.png"), dpi=150)
    
    # 2. All metrics comparison
    fig = plot_all_metrics(chunk_results)
    figures['all_metrics'] = fig
    if save_plots:
        fig.savefig(os.path.join(output_dir, f"{story_name}_all_metrics.png"), dpi=150)
    
    # 3. Metric correlations
    fig = plot_metric_correlation(chunk_results)
    figures['correlations'] = fig
    if save_plots:
        fig.savefig(os.path.join(output_dir, f"{story_name}_correlations.png"), dpi=150)
    
    print(f"✅ Generated visualizations for {story_name}")
    return figures

print("✅ Batch processing functions defined")

## 10. Demo: Process a Single Story

In [ ]:
# List available stories
story_files = sorted(glob(os.path.join(config.stories_dir, "*.txt")))
print(f"Found {len(story_files)} stories:")
for i, f in enumerate(story_files[:10]):
    print(f"  {i}: {Path(f).stem}")
if len(story_files) > 10:
    print(f"  ... and {len(story_files) - 10} more")

In [ ]:
# Process a single story (uncomment and modify as needed)
# story_path = story_files[0]  # Select first story
# chunk_results, story_analysis = process_story(story_path, verbose=True)

In [ ]:
# Generate visualizations (uncomment after processing)
# figures = generate_all_visualizations(chunk_results, story_analysis)

In [ ]:
# Export results (uncomment after processing)
# export_paths = export_all_formats(chunk_results, story_analysis, config.output_dir)
# print("\nExported files:")
# for fmt, path in export_paths.items():
#     print(f"  {fmt}: {path}")

## 11. Process Multiple Stories

In [ ]:
# Process multiple stories (uncomment and modify as needed)
# selected_stories = story_files[:5]  # Process first 5 stories
# all_chunks, all_analyses = process_multiple_stories(selected_stories, verbose=False)

In [ ]:
# Generate aggregate visualizations (uncomment after batch processing)
# if all_analyses:
#     # Heatmap across stories
#     fig_heatmap = plot_heatmap(all_analyses)
#     fig_heatmap.savefig(os.path.join(config.output_dir, "stories_heatmap.png"), dpi=150)
#     
#     # Histogram of surprise distribution
#     fig_hist = plot_histogram_by_genre(all_analyses)
#     fig_hist.savefig(os.path.join(config.output_dir, "surprise_distribution.png"), dpi=150)
#     
#     print("✅ Aggregate visualizations saved")

## 12. Utility Functions

In [ ]:
def load_results_from_jsonl(filepath: str) -> List[ChunkResult]:
    """
    Load chunk results from JSONL file.
    """
    results = []
    with open(filepath, 'r') as f:
        for line in f:
            data = json.loads(line.strip())
            results.append(ChunkResult(**data))
    return results


def load_results_from_parquet(filepath: str) -> List[ChunkResult]:
    """
    Load chunk results from Parquet file.
    """
    df = pd.read_parquet(filepath)
    
    # Parse embedding columns if they're stored as JSON strings
    if 'prediction_embedding' in df.columns:
        df['prediction_embedding'] = df['prediction_embedding'].apply(
            lambda x: json.loads(x) if isinstance(x, str) else x
        )
        df['actual_embedding'] = df['actual_embedding'].apply(
            lambda x: json.loads(x) if isinstance(x, str) else x
        )
    
    return [ChunkResult(**row) for row in df.to_dict('records')]


def compare_stories(analyses: List[StoryAnalysis]) -> pd.DataFrame:
    """
    Create a comparison DataFrame of story-level metrics.
    """
    data = []
    for a in analyses:
        data.append({
            'Story': a.story_title,
            'Words': a.total_words,
            'Chunks': a.total_chunks,
            'Avg Cosine Dist': round(a.mean_cosine_distance, 4),
            'Avg Euclidean Dist': round(a.mean_euclidean_distance, 4),
            'Avg KL Div': round(a.mean_kl_divergence, 4),
            'Detected Twists': a.num_detected_twists
        })
    
    df = pd.DataFrame(data)
    return df.sort_values('Avg Cosine Dist', ascending=False)

print("✅ Utility functions defined")

---

## Summary

This notebook provides a complete pipeline for quantifying narrative surprise:

### Features:
1. **BAU Prediction & Summary Generation**: Uses GPT-4o-mini to generate business-as-usual predictions and summaries
2. **OpenAI Embeddings**: Uses `text-embedding-3-large` for high-quality semantic embeddings
3. **Multiple Divergence Metrics**:
   - Cosine distance (standard semantic distance)
   - Euclidean distance (L2 norm in embedding space)
   - KL divergence (information-theoretic measure)
4. **Comprehensive Output**: Stores all data including raw embeddings
5. **Rich Visualizations**: Line plots, heatmaps, histograms, correlation matrices
6. **Multi-format Export**: CSV, JSONL, Parquet

### Usage:
```python
# Process a single story
chunks, analysis = process_story("path/to/story.txt")

# Generate visualizations
figures = generate_all_visualizations(chunks, analysis)

# Export results
export_all_formats(chunks, analysis, "./outputs")
```